# Lesson 5.2 — Encryption at Rest & In Transit: KMS, ACM & Secrets Manager

**Level: Intermediate** | **Exam:** SOA-C03 Domain 4, Task 4.2; SAP-C02 Domain 2, Task 2.3

## Learning Objectives
- Understand KMS key types (AWS managed, customer managed, imported) and when to use each
- Configure envelope encryption for S3, EBS, RDS, and DynamoDB
- Use AWS Certificate Manager (ACM) for TLS certificate management
- Store and rotate secrets using AWS Secrets Manager
- Implement data classification and encryption strategy for different data types

_Reference: [AWS KMS](https://docs.aws.amazon.com/kms/latest/developerguide/) | [Secrets Manager](https://docs.aws.amazon.com/secretsmanager/latest/userguide/) | [ACM](https://docs.aws.amazon.com/acm/latest/userguide/)_

## AWS KMS Key Types

| Key Type | Created by | Manage rotation | Cost | Use when |
|----------|-----------|----------------|------|----------|
| **AWS managed** (`aws/s3`, `aws/rds`) | AWS | AWS (auto, 1yr) | Free | Default, no special control needed |
| **Customer managed (CMK)** | You | You (manual or auto) | $1/key/month + usage | Need key policy, audit, cross-account sharing |
| **Customer managed + imported** | You (import material) | You | $1/key/month | Compliance requiring your own key material |
| **AWS CloudHSM** | You (HSM hardware) | You | $$$ | FIPS 140-2 Level 3 hardware requirement |

## Envelope Encryption

KMS uses **envelope encryption** — data is encrypted with a data key, and the data key is encrypted with the CMK:

```
1. App calls KMS: GenerateDataKey(CMK-ARN)
2. KMS returns: plaintext_data_key + encrypted_data_key
3. App encrypts data with plaintext_data_key
4. App stores: encrypted_data + encrypted_data_key
5. App discards plaintext_data_key from memory

Decrypt:
1. App calls KMS: Decrypt(encrypted_data_key) → plaintext_data_key
2. App decrypts data with plaintext_data_key
```

## Key Policy vs IAM Policy
- KMS **key policy** controls who can use and manage the key
- IAM **identity policy** can grant permission to use KMS (must be allowed in both!)
- Key policy is **required** — unlike S3 bucket policies, not optional

## Secrets Manager vs Parameter Store

| Feature | Secrets Manager | Parameter Store SecureString |
|---------|----------------|------------------------------|
| Cost | $0.40/secret/month | Free (standard) |
| Auto-rotation | Yes (built-in for RDS, Redshift) | No (manual or Lambda) |
| Cross-region replication | Yes | No |
| Max size | 65KB | 8KB (advanced) |
| Use for | DB passwords, API keys | Config + secrets, non-rotation |

In [ ]:
import boto3, json

kms = boto3.client('kms', region_name='us-east-1')
secrets = boto3.client('secretsmanager', region_name='us-east-1')
acm = boto3.client('acm', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Create a Customer Managed KMS Key

key_policy = {
    'Version': '2012-10-17',
    'Statement': [
        {
            'Sid': 'Enable IAM User Permissions',
            'Effect': 'Allow',
            'Principal': {'AWS': f'arn:aws:iam::{account_id}:root'},
            'Action': 'kms:*',
            'Resource': '*'
        },
        {
            'Sid': 'Allow CloudWatch Logs',
            'Effect': 'Allow',
            'Principal': {'Service': f'logs.us-east-1.amazonaws.com'},
            'Action': ['kms:Encrypt*', 'kms:Decrypt*', 'kms:ReEncrypt*',
                       'kms:GenerateDataKey*', 'kms:Describe*'],
            'Resource': '*',
            'Condition': {'ArnLike': {'kms:EncryptionContext:aws:logs:arn': f'arn:aws:logs:us-east-1:{account_id}:*'}}
        }
    ]
}

try:
    key = kms.create_key(
        Description='CloudOps Learning CMK - application data encryption',
        KeyUsage='ENCRYPT_DECRYPT',
        Origin='AWS_KMS',
        EnableKeyRotation=True,   # Auto-rotate annually
        Policy=json.dumps(key_policy),
        Tags=[{'TagKey': 'Purpose', 'TagValue': 'cloudops-learning'}]
    )
    KEY_ID = key['KeyMetadata']['KeyId']
    KEY_ARN = key['KeyMetadata']['Arn']
    print(f'Created KMS Key: {KEY_ID}')
    
    # Create a friendly alias
    kms.create_alias(AliasName='alias/cloudops-learning', TargetKeyId=KEY_ID)
    print(f'Created alias: alias/cloudops-learning → {KEY_ID}')
except kms.exceptions.AlreadyExistsException:
    keys = kms.list_aliases()
    for alias in keys['Aliases']:
        if alias.get('AliasName') == 'alias/cloudops-learning':
            KEY_ID = alias['TargetKeyId']
            KEY_ARN = f'arn:aws:kms:us-east-1:{account_id}:key/{KEY_ID}'
            print(f'Key exists: {KEY_ID}')
            break

In [ ]:
# Demonstrate envelope encryption
import base64

# 1. Generate a data key
data_key = kms.generate_data_key(
    KeyId='alias/cloudops-learning',
    KeySpec='AES_256'
)

plaintext_key = data_key['Plaintext']      # Use this to encrypt, then discard
encrypted_key = data_key['CiphertextBlob'] # Store this alongside your encrypted data

# 2. Encrypt some data with the data key (using Python cryptography)
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

# Generate nonce (initialization vector)
nonce = os.urandom(12)
cipher = AESGCM(plaintext_key)

payload = b'Sensitive data: database_password=SuperSecret123'
ciphertext = cipher.encrypt(nonce, payload, None)

print('Envelope Encryption Demo:')
print(f'  Original:        {payload}')
print(f'  Encrypted (hex): {ciphertext.hex()[:40]}...')
print(f'  Encrypted key:   {base64.b64encode(encrypted_key).decode()[:40]}...')
print()
print('Store on disk: [nonce + ciphertext + encrypted_data_key]')
print('The plaintext data key is never stored — only in memory briefly')

# 3. Decrypt (simulating a fresh process)
decrypted_key_response = kms.decrypt(CiphertextBlob=encrypted_key)
recovered_key = decrypted_key_response['Plaintext']

cipher2 = AESGCM(recovered_key)
decrypted = cipher2.decrypt(nonce, ciphertext, None)
print(f'\nDecrypted: {decrypted}')

In [ ]:
# AWS Secrets Manager — store and rotate database credentials

SECRET_NAME = 'cloudops-learning/db-credentials'

try:
    secret = secrets.create_secret(
        Name=SECRET_NAME,
        Description='Database credentials for CloudOps learning demo',
        SecretString=json.dumps({
            'engine': 'postgres',
            'host': 'myapp-prod.cluster-xyz.us-east-1.rds.amazonaws.com',
            'username': 'appuser',
            'password': 'InitialPassword123!',  # Would be rotated automatically
            'dbname': 'appdb',
            'port': 5432
        }),
        KmsKeyId='alias/cloudops-learning'
    )
    print(f'Created secret: {secret["ARN"]}')
except secrets.exceptions.ResourceExistsException:
    secret = secrets.describe_secret(SecretId=SECRET_NAME)
    print(f'Secret exists: {secret["ARN"]}')

# Retrieve the secret (how your application does it)
value = secrets.get_secret_value(SecretId=SECRET_NAME)
db_creds = json.loads(value['SecretString'])
print(f'\nRetrieved credentials for host: {db_creds["host"]}')
print(f'User: {db_creds["username"]} / DB: {db_creds["dbname"]}')
print('Password: ***MASKED***')

In [ ]:
# ACM — AWS Certificate Manager
# Lists existing certificates (you can't create self-signed certs via API — need domain validation)

certs = acm.list_certificates()['CertificateSummaryList']
print(f'ACM Certificates ({len(certs)}):')
for cert in certs:
    print(f'  {cert["CertificateArn"]}  {cert["DomainName"]}  {cert.get("Status", "UNKNOWN")}')

if not certs:
    print('No certificates found.')
    print('\nRequest a public certificate (requires domain ownership validation):')
    print('acm.request_certificate(')
    print('    DomainName="example.com",')
    print('    ValidationMethod="DNS",    # or EMAIL')
    print('    SubjectAlternativeNames=["*.example.com", "www.example.com"]')
    print(')')
    print('\nCertificates are FREE for use with ALB, CloudFront, API Gateway')

## Exercise

In [ ]:
# TODO: Audit KMS usage in your account:
# 1. List all customer-managed KMS keys (not AWS-managed)
# 2. For each key, check:
#    - Is automatic rotation enabled?
#    - What is the key policy (can any external principals access it?)
#    - When was the key last rotated?
# 3. Flag any key with rotation disabled

import boto3

def audit_kms_keys():
    kms = boto3.client('kms', region_name='us-east-1')
    issues = []
    
    # YOUR CODE HERE
    
    return issues

results = audit_kms_keys()
print(f'KMS Audit: {len(results)} issues found')
for r in results:
    print(f'  {r}')

In [ ]:
# Cleanup
import boto3
kms = boto3.client('kms', region_name='us-east-1')
secrets = boto3.client('secretsmanager', region_name='us-east-1')

try:
    secrets.delete_secret(SecretId='cloudops-learning/db-credentials', ForceDeleteWithoutRecovery=True)
    print('Deleted secret')
except Exception as e:
    print(f'Secret: {e}')

try:
    kms.delete_alias(AliasName='alias/cloudops-learning')
    kms.schedule_key_deletion(KeyId=KEY_ID, PendingWindowInDays=7)
    print(f'Scheduled key deletion (7-day pending window): {KEY_ID}')
except Exception as e:
    print(f'KMS cleanup: {e}')

## Summary
- **KMS key types**: AWS managed (free, auto-rotate), Customer managed ($1/mo, full control), CloudHSM (hardware compliance)
- **Envelope encryption**: Data Key (encrypt data) + Master Key (encrypt data key) — never store plaintext keys
- **Enable automatic key rotation** on all CMKs — yearly rotation by default
- **Secrets Manager**: Auto-rotation for RDS/Redshift; use for passwords and API keys
- **Parameter Store SecureString**: Cheaper, no auto-rotation; good for non-password config secrets
- **ACM**: Free TLS certificates for ALB, CloudFront, API Gateway

---
**Next:** [Lesson 5.3 — Compliance: AWS Config, Security Hub, GuardDuty & Inspector](lesson-5.3-compliance-tools.ipynb)